# Notebook 08: Activity-Adaptive Anatomical Graph Neural Network (Adaptive GNN)

**Objective**: Train and evaluate the core research model — the Activity-Adaptive Anatomical Graph Neural Network (`ActivityAdaptiveAnatomicalGNN`).

### Core Research Hypothesis:
Different activities (walking, sitting, standing, reaching) induce dynamic, task-dependent functional coupling between limbs. An adaptive graph formulation where edge weights depend on the active input window $X$ captures these relationships more effectively than a fixed, static skeletal prior.

### Dynamic Graph Formulation:
$$A_{\text{dyn}} = (1 - \lambda) \tilde{A}_{\text{fixed}} + \lambda A_{\text{adapt}}(h_0)$$
where $\lambda = \sigma(w_\lambda)$ balances the documented anatomical skeletal prior $\tilde{A}_{\text{fixed}}$ with the learned pairwise attention affinity $A_{\text{adapt}}$.

> **Compute Rule**: Full training requires CUDA GPU (Colab). Set `EXPERIMENT_MODE = 'DEBUG'` for 1-batch CPU verification.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT DETECTION & REPOSITORY BOOTSTRAP
# ==============================================================================
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("[INFO] Cloning Opportunity_HAR repository from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    else:
        print(f"[INFO] Repository already present at {REPO_DIR}. Pulling latest changes...")
        !git -C {REPO_DIR} pull origin main

    # Set repository root at front of sys.path
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

print(f"Active Working Directory : {Path.cwd()}")

# Ensure fresh import of src.api
import importlib
if "src.api" in sys.modules:
    import src.api
    importlib.reload(src.api)
else:
    import src.api as api

import src.api as api
status = api.setup_project()
print(f"High-Level API Loaded    : src.api (locate_dataset available: {hasattr(api, 'locate_dataset')})")
print(f"Compute Device Detected  : {status['device_name']} (CUDA Available: {status['cuda_available']})")

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'DEBUG' (1-epoch CPU/GPU check), 'FAST' (5 epochs GPU), or 'FULL' (50 epochs GPU)
EXPERIMENT_MODE = "FAST" if env_info["cuda_available"] else "DEBUG"
print(f"Selected EXPERIMENT_MODE: {EXPERIMENT_MODE}")
if not env_info["cuda_available"] and EXPERIMENT_MODE != "DEBUG":
    print("Notice: CUDA is not available. Please switch Colab to GPU (Runtime > Change runtime type > T4 GPU) or use DEBUG mode.")

## 2. Load Processed Sliding Windows & DataLoaders

In [ ]:
try:
    subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
    X_train, y_train = subset["X_train"], subset["y_train"]
    X_val, y_val = subset["X_val"], subset["y_val"]
    X_test, y_test = subset["X_test"], subset["y_test"]

    if EXPERIMENT_MODE == "DEBUG":
        X_train, y_train = X_train[:64], y_train[:64]
        X_val, y_val = X_val[:32], y_val[:32]
        X_test, y_test = X_test[:32], y_test[:32]

    train_loader, val_loader, test_loader = api.create_dataloaders(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
    )
    print(f"DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches.")
except FileNotFoundError:
    print("[NOTICE] Preprocessed dataset not found. Run Notebook 02 to generate opportunity_locomotion_subset.npz.")

## 3. Instantiate Activity-Adaptive Anatomical GNN

In [ ]:
adaptive_gnn = api.build_adaptive_gnn(
    num_classes=5,
    sequence_length=30,
    node_dim=64,
    gcn_hidden_dim=64,
    dropout=0.2
)

total_params = sum(p.numel() for p in adaptive_gnn.parameters() if p.requires_grad)
print(f"ActivityAdaptiveAnatomicalGNN Initialized! Trainable parameters: {total_params:,}")

## 4. Train Model (Automatic GPU/CUDA Guard)

In [ ]:
try:
    best_adaptive_gnn, history = api.train_model(
        model=adaptive_gnn,
        train_loader=train_loader,
        val_loader=val_loader,
        mode=EXPERIMENT_MODE,
        lr=1e-3,
        experiment_name="adaptive_anatomical_gnn"
    )
    print("Training Complete!")
except Exception as e:
    print("Training message:", e)

## 5. Evaluate on Held-Out Test Set

In [ ]:
try:
    test_metrics = api.evaluate_model(best_adaptive_gnn, test_loader)
    print("\n" + "=" * 55)
    print(" ACTIVITY-ADAPTIVE ANATOMICAL GNN EVALUATION")
    print("=" * 55)
    print(f"Accuracy    : {test_metrics['accuracy'] * 100:.2f}%")
    print(f"Macro F1    : {test_metrics['macro_f1'] * 100:.2f}%")
    print(f"Weighted F1 : {test_metrics['weighted_f1'] * 100:.2f}%")
    print("=" * 55)
except Exception as e:
    print("Evaluation message:", e)

## 6. Inspect & Visualize Learned Dynamic Adjacency Matrices
Extract and plot the learned inter-region edge weights $A_{\text{dyn}}$ for sample activity windows.

In [ ]:
try:
    import torch
    import matplotlib.pyplot as plt
    import seaborn as sns

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_adaptive_gnn.eval()

    sample_x = torch.from_numpy(X_test[:4]).float().to(device)
    with torch.no_grad():
        logits, dynamic_adj = best_adaptive_gnn(sample_x, return_attention=True)

    adj_matrices = dynamic_adj.cpu().numpy()
    region_names = [r.replace("_", " ").title() for r in BODY_REGIONS]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for i in range(2):
        sns.heatmap(
            adj_matrices[i],
            annot=True, fmt=".2f", cmap="YlGnBu",
            xticklabels=region_names, yticklabels=region_names,
            ax=axes[i], vmin=0.0, vmax=0.6
        )
        axes[i].set_title(f"Dynamic Adjacency A_dyn (Window #{i})")
        axes[i].set_xlabel("Target Region")
        axes[i].set_ylabel("Source Region")

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Visualization message:", e)